In [1]:

import pyspark

from pyspark.sql import SparkSession
s=SparkSession.builder.appName('ecommerce').getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/07 15:18:59 WARN Utils: Your hostname, swalih-Victus-by-HP-Gaming-Laptop-15-fb0xxx, resolves to a loopback address: 127.0.1.1; using 10.193.184.236 instead (on interface wlo1)
26/03/07 15:18:59 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/07 15:19:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/07 15:19:01 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:


df = s.read.csv(
    "ecommerce_dirty_dataset.csv",  # file name
    header=True,        # Use first row as column names
    inferSchema=True    # Automatically detect column data types
)

In [3]:
df.show()

+--------+-------------+---------+----------+--------+------+
|order_id|customer_name|     city|   product|quantity| price|
+--------+-------------+---------+----------+--------+------+
|       1|         NULL|  Calicut|     Phone|     5.0|1569.0|
|       2|         Neha|Bangalore|Headphones|     4.0|1616.0|
|       3|        Rahul|    Delhi|   Monitor|     5.0|1525.0|
|       4|        Kiran|  Chennai|  Keyboard|    NULL|1022.0|
|       5|        Ahmed|    Delhi|    Camera|     3.0| 264.0|
|       6|          Anu|    Kochi|    Tablet|     3.0|  NULL|
|       7|        Aisha|    Delhi|  Keyboard|     1.0| 698.0|
|       8|         Neha|    Kochi|      NULL|     3.0| 954.0|
|       9|        Priya|Bangalore|Headphones|     1.0| 350.0|
|      10|         Neha|  Chennai|    Tablet|     3.0|1980.0|
|      11|        Kiran|Bangalore|  Keyboard|     3.0| 861.0|
|      12|        Sneha|    Kochi|     Phone|     3.0|1235.0|
|      13|        Priya|   Mumbai|     Phone|     3.0| 144.0|
|      1

In [4]:
from pyspark.sql.functions import col


# col() allows us to reference a column inside transformations.

# col("price")

In [5]:
# Check Missing Values

df.select(
        [col(c).isNull().alias(c)
           for c in df.columns]
          ).show()

+--------+-------------+-----+-------+--------+-----+
|order_id|customer_name| city|product|quantity|price|
+--------+-------------+-----+-------+--------+-----+
|   false|         true|false|  false|   false|false|
|   false|        false|false|  false|   false|false|
|   false|        false|false|  false|   false|false|
|   false|        false|false|  false|    true|false|
|   false|        false|false|  false|   false|false|
|   false|        false|false|  false|   false| true|
|   false|        false|false|  false|   false|false|
|   false|        false|false|   true|   false|false|
|   false|        false|false|  false|   false|false|
|   false|        false|false|  false|   false|false|
|   false|        false|false|  false|   false|false|
|   false|        false|false|  false|   false|false|
|   false|        false|false|  false|   false|false|
|   false|        false|false|  false|   false|false|
|   false|        false|false|  false|   false|false|
|   false|        false|fals

In [6]:
from pyspark.sql.functions import sum

df.select(
    [col(c).isNull().cast('int').alias(c)
     for c in df.columns]
).show()

+--------+-------------+----+-------+--------+-----+
|order_id|customer_name|city|product|quantity|price|
+--------+-------------+----+-------+--------+-----+
|       0|            1|   0|      0|       0|    0|
|       0|            0|   0|      0|       0|    0|
|       0|            0|   0|      0|       0|    0|
|       0|            0|   0|      0|       1|    0|
|       0|            0|   0|      0|       0|    0|
|       0|            0|   0|      0|       0|    1|
|       0|            0|   0|      0|       0|    0|
|       0|            0|   0|      1|       0|    0|
|       0|            0|   0|      0|       0|    0|
|       0|            0|   0|      0|       0|    0|
|       0|            0|   0|      0|       0|    0|
|       0|            0|   0|      0|       0|    0|
|       0|            0|   0|      0|       0|    0|
|       0|            0|   0|      0|       0|    0|
|       0|            0|   0|      0|       0|    0|
|       0|            0|   0|      0|       0|

In [7]:
# Count Missing Values 

df.select([sum(col(c).isNull().cast('int').alias(c))
           for c in df.columns]).show()

+------------------------------------------------+----------------------------------------------------------+----------------------------------------+----------------------------------------------+------------------------------------------------+------------------------------------------+
|sum(CAST((order_id IS NULL) AS INT) AS order_id)|sum(CAST((customer_name IS NULL) AS INT) AS customer_name)|sum(CAST((city IS NULL) AS INT) AS city)|sum(CAST((product IS NULL) AS INT) AS product)|sum(CAST((quantity IS NULL) AS INT) AS quantity)|sum(CAST((price IS NULL) AS INT) AS price)|
+------------------------------------------------+----------------------------------------------------------+----------------------------------------+----------------------------------------------+------------------------------------------------+------------------------------------------+
|                                               0|                                                         8|                     

In [8]:
from pyspark.sql.functions import sum

df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).show()

+--------+-------------+----+-------+--------+-----+
|order_id|customer_name|city|product|quantity|price|
+--------+-------------+----+-------+--------+-----+
|       0|            8|   7|      6|       6|    8|
+--------+-------------+----+-------+--------+-----+



In [9]:
# Remove Rows Where Target is NULL

df=df.dropna(subset='price')

In [10]:
df.select([sum(col('price').isNull().cast('int')).alias('price')]
          ).show()

+-----+
|price|
+-----+
|    0|
+-----+



In [11]:
from pyspark.sql.functions import sum

df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).show()

+--------+-------------+----+-------+--------+-----+
|order_id|customer_name|city|product|quantity|price|
+--------+-------------+----+-------+--------+-----+
|       0|            8|   7|      5|       6|    0|
+--------+-------------+----+-------+--------+-----+



In [12]:
## Fill Missing Quantity
# Quantity is numeric, so we fill with median or mean.

from pyspark.sql.functions import avg


mean_quantity=df.select(avg('quantity'))

In [13]:
mean_quantity.show()

+-----------------+
|    avg(quantity)|
+-----------------+
|3.020408163265306|
+-----------------+



In [14]:
mean_quantity=df.select(avg('quantity')).collect()[0][0]

In [15]:
mean_quantity

3.020408163265306

In [16]:
mean_quantity=int(mean_quantity)

In [17]:
mean_quantity

3

In [18]:
df=df.fillna({'quantity':mean_quantity})

In [19]:
# Step 6 — Fill Missing Product

# Product is categorical.

# We fill with most frequent value (mode).

In [20]:
most_product=df.groupBy('product').count().orderBy('count',ascending=False)

most_product.show()

+----------+-----+
|   product|count|
+----------+-----+
|Headphones|   19|
|     Phone|   18|
|    Laptop|   18|
|  Keyboard|   13|
|    Tablet|   11|
|   Monitor|   11|
|    Camera|    9|
|      NULL|    5|
+----------+-----+



In [21]:
most_product=df.groupBy('product').count().orderBy('count',ascending=False).first()

most_product

Row(product='Headphones', count=19)

In [22]:
most_product=df.groupBy('product').count().orderBy('count',ascending=False).first()[0]

most_product

'Headphones'

In [23]:
df=df.fillna({'product':most_product})

In [24]:
# Fill Missing Customer Name , city

## Customer name is not useful for prediction, but we can fill it.

In [25]:
df = df.fillna({"customer_name": "Unknown",'city':'Unknown'})



In [26]:
# Verify Cleaning


In [27]:
df.select([sum(col(c).isNull().cast('int')).alias(c)
           for c in df.columns]).show()

+--------+-------------+----+-------+--------+-----+
|order_id|customer_name|city|product|quantity|price|
+--------+-------------+----+-------+--------+-----+
|       0|            0|   0|      0|       0|    0|
+--------+-------------+----+-------+--------+-----+



# BEST WAY

In [28]:
cat_cols = [c for c,t in df.dtypes if t == 'string']
num_cols = [c for c,t in df.dtypes if t != 'string']

In [29]:
# approxQuantile() is a Spark function used to calculate quantiles (percentiles) efficiently on big data.

# df.approxQuantile(column, probabilities, relativeError)[position]

for col_name in num_cols:
    
    median = df.approxQuantile(col_name,[0.5],0)[0]
    
    df = df.fillna({col_name: median})

In [30]:
for col_name in cat_cols:
    
    mode = df.groupBy(col_name).count()\
             .orderBy("count", ascending=False)\
             .first()[0]
    
    df = df.fillna({col_name: mode})

In [31]:
from pyspark.sql.functions import col,sum

df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).show()

+--------+-------------+----+-------+--------+-----+
|order_id|customer_name|city|product|quantity|price|
+--------+-------------+----+-------+--------+-----+
|       0|            0|   0|      0|       0|    0|
+--------+-------------+----+-------+--------+-----+



In [32]:
df=df.drop('customer_name')

In [33]:
from pyspark.sql.functions import col,sum

df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).show()

+--------+----+-------+--------+-----+
|order_id|city|product|quantity|price|
+--------+----+-------+--------+-----+
|       0|   0|      0|       0|    0|
+--------+----+-------+--------+-----+



## Feature Encoding (Convert Text → Numbers)

In [34]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder

In [35]:
# Convert Categories into Index Numbers


indexer=StringIndexer(
    inputCols=['city','product'],
    outputCols=['city1','product1']
)

In [36]:
df=indexer.fit(df).transform(df)

In [37]:
df.show()

+--------+---------+----------+--------+------+-----+--------+
|order_id|     city|   product|quantity| price|city1|product1|
+--------+---------+----------+--------+------+-----+--------+
|       1|  Calicut|     Phone|     5.0|1569.0|  5.0|     2.0|
|       2|Bangalore|Headphones|     4.0|1616.0|  3.0|     0.0|
|       3|    Delhi|   Monitor|     5.0|1525.0|  0.0|     4.0|
|       4|  Chennai|  Keyboard|     3.0|1022.0|  7.0|     3.0|
|       5|    Delhi|    Camera|     3.0| 264.0|  0.0|     6.0|
|       7|    Delhi|  Keyboard|     1.0| 698.0|  0.0|     3.0|
|       8|    Kochi|Headphones|     3.0| 954.0|  1.0|     0.0|
|       9|Bangalore|Headphones|     1.0| 350.0|  3.0|     0.0|
|      10|  Chennai|    Tablet|     3.0|1980.0|  7.0|     5.0|
|      11|Bangalore|  Keyboard|     3.0| 861.0|  3.0|     3.0|
|      12|    Kochi|     Phone|     3.0|1235.0|  1.0|     2.0|
|      13|   Mumbai|     Phone|     3.0| 144.0|  4.0|     2.0|
|      14|    Delhi|Headphones|     2.0| 691.0|  0.0|  

In [38]:
encoder = OneHotEncoder(
    inputCols=["city1","product1"],
    outputCols=["city_vec","product_vec"]
)

In [39]:
df=encoder.fit(df).transform(df)

In [40]:
df.show()

+--------+---------+----------+--------+------+-----+--------+-------------+-------------+
|order_id|     city|   product|quantity| price|city1|product1|     city_vec|  product_vec|
+--------+---------+----------+--------+------+-----+--------+-------------+-------------+
|       1|  Calicut|     Phone|     5.0|1569.0|  5.0|     2.0|(7,[5],[1.0])|(6,[2],[1.0])|
|       2|Bangalore|Headphones|     4.0|1616.0|  3.0|     0.0|(7,[3],[1.0])|(6,[0],[1.0])|
|       3|    Delhi|   Monitor|     5.0|1525.0|  0.0|     4.0|(7,[0],[1.0])|(6,[4],[1.0])|
|       4|  Chennai|  Keyboard|     3.0|1022.0|  7.0|     3.0|    (7,[],[])|(6,[3],[1.0])|
|       5|    Delhi|    Camera|     3.0| 264.0|  0.0|     6.0|(7,[0],[1.0])|    (6,[],[])|
|       7|    Delhi|  Keyboard|     1.0| 698.0|  0.0|     3.0|(7,[0],[1.0])|(6,[3],[1.0])|
|       8|    Kochi|Headphones|     3.0| 954.0|  1.0|     0.0|(7,[1],[1.0])|(6,[0],[1.0])|
|       9|Bangalore|Headphones|     1.0| 350.0|  3.0|     0.0|(7,[3],[1.0])|(6,[0],[1.0])|

##  Combine Features

ML models need one feature column.

PySpark uses VectorAssembler.

In [41]:
from pyspark.ml.feature import VectorAssembler

assembler=VectorAssembler(
    inputCols=['city_vec','product_vec','quantity'],
    outputCol='features'
)

In [42]:
df=assembler.transform(df)

In [44]:
df.show()

+--------+---------+----------+--------+------+-----+--------+-------------+-------------+--------------------+
|order_id|     city|   product|quantity| price|city1|product1|     city_vec|  product_vec|            features|
+--------+---------+----------+--------+------+-----+--------+-------------+-------------+--------------------+
|       1|  Calicut|     Phone|     5.0|1569.0|  5.0|     2.0|(7,[5],[1.0])|(6,[2],[1.0])|(14,[5,9,13],[1.0...|
|       2|Bangalore|Headphones|     4.0|1616.0|  3.0|     0.0|(7,[3],[1.0])|(6,[0],[1.0])|(14,[3,7,13],[1.0...|
|       3|    Delhi|   Monitor|     5.0|1525.0|  0.0|     4.0|(7,[0],[1.0])|(6,[4],[1.0])|(14,[0,11,13],[1....|
|       4|  Chennai|  Keyboard|     3.0|1022.0|  7.0|     3.0|    (7,[],[])|(6,[3],[1.0])|(14,[10,13],[1.0,...|
|       5|    Delhi|    Camera|     3.0| 264.0|  0.0|     6.0|(7,[0],[1.0])|    (6,[],[])|(14,[0,13],[1.0,3...|
|       7|    Delhi|  Keyboard|     1.0| 698.0|  0.0|     3.0|(7,[0],[1.0])|(6,[3],[1.0])|(14,[0,10,13],

## Train Test Split

In [45]:
train_df,test_df=df.randomSplit([.8,.2],seed=42)

In [47]:
train_df.count()


86

In [48]:
test_df.count()

18

## Create Model Object

In [49]:
from pyspark.ml.regression import LinearRegression

lr=LinearRegression(featuresCol='features',
                    labelCol='price')

In [ ]:
model = lr.fit(train_df)

26/03/07 15:31:38 WARN Instrumentation: [292f8446] regParam is zero, which might cause numerical instability and overfitting.
26/03/07 15:31:38 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/03/07 15:31:38 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


In [51]:
predictions = model.transform(test_df)

In [52]:
predictions.show()

+--------+---------+----------+--------+------+-----+--------+-------------+-------------+--------------------+------------------+
|order_id|     city|   product|quantity| price|city1|product1|     city_vec|  product_vec|            features|        prediction|
+--------+---------+----------+--------+------+-----+--------+-------------+-------------+--------------------+------------------+
|       2|Bangalore|Headphones|     4.0|1616.0|  3.0|     0.0|(7,[3],[1.0])|(6,[0],[1.0])|(14,[3,7,13],[1.0...| 823.4593910542841|
|       7|    Delhi|  Keyboard|     1.0| 698.0|  0.0|     3.0|(7,[0],[1.0])|(6,[3],[1.0])|(14,[0,10,13],[1....| 799.0859715803722|
|       9|Bangalore|Headphones|     1.0| 350.0|  3.0|     0.0|(7,[3],[1.0])|(6,[0],[1.0])|(14,[3,7,13],[1.0...| 655.7555444351135|
|      13|   Mumbai|     Phone|     3.0| 144.0|  4.0|     2.0|(7,[4],[1.0])|(6,[2],[1.0])|(14,[4,9,13],[1.0...| 781.3851989640531|
|      19|Bangalore|    Laptop|     4.0|1217.0|  3.0|     1.0|(7,[3],[1.0])|(6,[1],

## Evaluate Model Performance

In [53]:
from pyspark.ml.evaluation import RegressionEvaluator
evaluator = RegressionEvaluator(
    labelCol="price",
    predictionCol="prediction",
    metricName="rmse"
)

In [54]:
rmse = evaluator.evaluate(predictions)

print("RMSE:", rmse)

RMSE: 463.51486940038944


In [55]:
evaluator = RegressionEvaluator(
    labelCol="price",
    predictionCol="prediction",
    metricName="r2"
)

r2 = evaluator.evaluate(predictions)

print("R2:", r2)

R2: 0.12887499107346412


In [56]:
# this is example model building there is no such features that effect price that why its very low r2 score